# Một số thư viện 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_validate, GridSearchCV, RandomizedSearchCV

# Đọc file 

In [ ]:
behavior = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAIN_QUANTITATIVE_METADATA_new.xlsx")
demographics = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAIN_CATEGORICAL_METADATA_new.xlsx")
fmri = pd.read_csv(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
labels = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TRAIN_NEW\TRAINING_SOLUTIONS.xlsx")

behavior_test = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TEST\TEST_QUANTITATIVE_METADATA.xlsx")
demographics_test = pd.read_excel(r"C:\Users\DELL\Downloads\widsdatathon2025\TEST\TEST_CATEGORICAL.xlsx")
fmri_test = pd.read_csv(r"C:\Users\DELL\Downloads\widsdatathon2025\TEST\TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")

# Tiền xử lý dữ liệu

## Dữ liệu hành vi 

In [ ]:
behavior_features = behavior.drop(columns=['participant_id'])

imputer = SimpleImputer(strategy="mean")
behavior_imputed=imputer.fit_transform(behavior_features)

scaler = StandardScaler()
behavior_scaled = scaler.fit_transform(behavior_imputed)

## Dữ liệu nhân khẩu học 

In [ ]:
# Tách participant_id
demographics_features = demographics.drop(columns=['participant_id'])

# Phân loại feature
numerical_features = ['Basic_Demos_Enroll_Year']
ordinal_features = [
    'Barratt_Barratt_P1_Edu', 'Barratt_Barratt_P1_Occ',
    'Barratt_Barratt_P2_Edu', 'Barratt_Barratt_P2_Occ'
]
categorical_features = [
    'Basic_Demos_Study_Site',
    'PreInt_Demos_Fam_Child_Ethnicity',
    'PreInt_Demos_Fam_Child_Race',
    'MRI_Track_Scan_Location'
]

# Pipeline cho từng nhóm
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),   # Điền thiếu bằng mean
    ('scaler', StandardScaler())                   # Chuẩn hóa
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Điền thiếu bằng mode
    ('scaler', StandardScaler())                           # Chuẩn hóa vì giá trị có thứ tự
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Điền thiếu bằng mode
    ('encoder', OneHotEncoder(handle_unknown='ignore'))    # One-hot encode
])

# Gộp tất cả lại
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('ord', ordinal_pipeline, ordinal_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Sau đó dùng preprocessor cho X
demographics_preprocessed = preprocessor.fit_transform(demographics_features)

## Dữ liệu MRI 

In [ ]:
# Tách participant_id
fmri_features = fmri.drop(columns=['participant_id'])

# Pipeline xử lý MRI data
mri_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),    # Điền NaN bằng mean
    ('scaler', StandardScaler()),                   # Chuẩn hóa
    ('pca', PCA(n_components=0.95))                  # Giữ 95% phương sai
])

# Apply pipeline
fmri_processed = mri_pipeline.fit_transform(fmri_features)

## Gộp theo `participant_id` 

In [ ]:
# Lấy participant_id
participant_ids = behavior['participant_id']

# Chuyển behavior_scaled, demographics_preprocessed, fmri_processed về dataframe để merge
behavior_scaled_df = pd.DataFrame(behavior_scaled, columns=behavior.columns.drop('participant_id'))
behavior_scaled_df['participant_id'] = participant_ids

demographics_preprocessed_df = pd.DataFrame(demographics_preprocessed, columns=[f'cat_{i}' for i in range(demographics_preprocessed.shape[1])])
demographics_preprocessed_df['participant_id'] = participant_ids

fmri_processed_df = pd.DataFrame(fmri_processed, columns=[f'mri_{i}' for i in range(fmri_processed.shape[1])])
fmri_processed_df['participant_id'] = participant_ids

# Gộp lại thành 1 dataframe đầy đủ
df = behavior_scaled_df.merge(demographics_preprocessed_df, on="participant_id") \
                        .merge(fmri_processed_df, on="participant_id") \
                        .merge(labels, on="participant_id")

## Chọn dữ liệu đầu vào 

In [ ]:
df_sample = df.sample(frac=1, random_state=42)

print(f"Tổng số dòng trong df: {len(df)}")
print(f"Số dòng sau khi sample 100%: {len(df_sample)}")

X = df_sample.drop(columns=["participant_id", "ADHD_Outcome", "Sex_F"])
y_adhd = df_sample["ADHD_Outcome"]
y_sex = df_sample["Sex_F"]

## Dùng SMOTE để cân bằng dữ liệu 

In [ ]:
smote = SMOTE(random_state=42)
X_adhd_resampled, y_adhd_resampled = smote.fit_resample(X, y_adhd)

X_train, X_test, y_train, y_test = train_test_split(X_adhd_resampled, y_adhd_resampled, test_size=0.2, random_state=42)

print("\nADHD Test Label Distribution:", np.bincount(y_test))

# Train models 

## Train models dự đoán ADHD 

In [ ]:
clf_adhd = RandomForestClassifier(random_state=42, class_weight="balanced")

'''
# 2. Đánh giá model mặc định (tùy chọn, có thể bỏ nếu muốn tiết kiệm thời gian)
cv_adhd_results = cross_validate(
    clf_adhd, X_train, y_train,
    cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1']
)
print("=== Cross Validation với model mặc định ===")
print("Mean Accuracy:", np.mean(cv_adhd_results['test_accuracy']))
print("Mean Precision:", np.mean(cv_adhd_results['test_precision']))
print("Mean Recall:", np.mean(cv_adhd_results['test_recall']))
print("Mean F1:", np.mean(cv_adhd_results['test_f1']))
'''

# 3. Grid Search để tìm hyperparameters tốt nhất
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

random_search_adhd = RandomizedSearchCV(
    estimator=clf_adhd,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

random_search_adhd.fit(X_train, y_train)

print("\n=== Kết quả Grid Search ===")
print("Best Hyperparameters:", random_search_adhd.best_params_)
print("Best CV Score (F1):", random_search_adhd.best_score_)

# 4. Dùng model tốt nhất để predict trên tập test
best_model_adhd = random_search_adhd.best_estimator_
y_pred_adhd = best_model_adhd.predict(X_test)

print("==== ADHD REPORT ====")
print(classification_report(y_test, y_pred_adhd))

print("\nSố mẫu ADHD:")
print(df_sample["ADHD_Outcome"].value_counts())

## Train model Sex 

In [ ]:
# Dùng kỹ thuật như SMOTE để tạo thêm dữ liệu ảo cho nhóm ADHD = 0 (hoặc Sex = 1)
smote = SMOTE(random_state=42)
X_sex_resampled, y_sex_resampled = smote.fit_resample(X, y_sex)

X_train, X_test, y_train, y_test = train_test_split(X_sex_resampled, y_sex_resampled, test_size=0.2, random_state=42)

print("\nSex Test Label Distribution:", np.bincount(y_test))

clf_sex = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")

# 3. Grid Search để tìm hyperparameters tốt nhất
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

random_search_sex = RandomizedSearchCV(
    estimator=clf_sex,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

random_search_sex.fit(X_train, y_train)

print("\n=== Kết quả Grid Search ===")
print("Best Hyperparameters:", random_search_sex.best_params_)
print("Best CV Score (F1):", random_search_sex.best_score_)

# 4. Dùng model tốt nhất để predict trên tập test
best_model_sex = random_search_sex.best_estimator_
y_pred_sex = best_model_sex.predict(X_test)

'''
cv_sex_results = cross_validate(
    clf_sex, X_train, y_train,
    cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1']
)

print("Mean Accuracy:", np.mean(cv_sex_results['test_accuracy']))
print("Mean Precision:", np.mean(cv_sex_results['test_precision']))
print("Mean Recall:", np.mean(cv_sex_results['test_recall']))
print("Mean F1:", np.mean(cv_sex_results['test_f1']))
'''

print("==== SEX REPORT ====")
print(classification_report(y_test, y_pred_sex))

print("\nSố mẫu SEX:")
print(df_sample["Sex_F"].value_counts())

# Tập dữ liệu test 

## Tiền xử lý như tập train 

In [ ]:
# Preprocess test giống như train

participant_ids_test = behavior_test['participant_id']

# Áp dụng preprocessor y hệt train
behavior_test_imputed = imputer.transform(behavior_test.drop(columns=["participant_id"]))
behavior_test_scaled = scaler.transform(behavior_test_imputed)

demographics_test_preprocessed = preprocessor.transform(demographics_test.drop(columns=["participant_id"]))
fmri_test_processed = mri_pipeline.transform(fmri_test.drop(columns=["participant_id"]))

# Gộp thành dataframe
behavior_test_df = pd.DataFrame(behavior_test_scaled, columns=behavior.columns.drop('participant_id'))
behavior_test_df['participant_id'] = participant_ids_test

demographics_test_df = pd.DataFrame(demographics_test_preprocessed, columns=[f'cat_{i}' for i in range(demographics_test_preprocessed.shape[1])])
demographics_test_df['participant_id'] = participant_ids_test

fmri_test_df = pd.DataFrame(fmri_test_processed, columns=[f'mri_{i}' for i in range(fmri_test_processed.shape[1])])
fmri_test_df['participant_id'] = participant_ids_test

df_test = behavior_test_df.merge(demographics_test_df, on="participant_id") \
                          .merge(fmri_test_df, on="participant_id")

## Dự đoán trên tập test 

In [ ]:
# === 8. DỰ ĐOÁN TRÊN DATA TEST ===
X_test_real = df_test.drop(columns=["participant_id"])

y_pred_adhd_test = best_model_adhd.predict(X_test_real)
y_pred_sex_test = best_model_sex.predict(X_test_real)

# === 9. LƯU FILE SUBMIT ===
df_submit = pd.DataFrame({
    "participant_id": df_test["participant_id"],
    "ADHD_Outcome": y_pred_adhd_test,
    "Sex_F": y_pred_sex_test
})

df_submit.to_csv(r"C:\Users\DELL\Downloads\widsdatathon2025\submission.csv", index=False)
print("Đã lưu kết quả dự đoán vào submission.csv!")